**1. `@pytest.mark.parametrize` — From Basic to Advanced**

**Level 1 — Single Parameter**

In [ ]:
import pytest

def add(a, b):
    return a + b

@pytest.mark.parametrize("a, b, expected", [
    (1,  2,  3),      # test case 1
    (0,  0,  0),      # test case 2
    (-1, 1,  0),      # test case 3
    (-1, -1, -2),     # test case 4
])
def test_add(a, b, expected):
    assert add(a, b) == expected

```shell
pytest runs 4 independent tests:
  test_add[1-2-3]
  test_add[0-0-0]
  test_add[-1-1-0]
  test_add[-1--1--2]
```

---

**Level 2 — Custom IDs (critical for readability)**

In [ ]:
@pytest.mark.parametrize(
    "a, b, expected", [
    (1,  2,  3),
    (-1, -1, -2),
    (0,  0,  0),
    ],
    ids=[
        "positive numbers",
        "negative numbers",
        "zeros",
    ]
)
def test_add_with_ids(a, b, expected):
    assert add(a, b) == expected

```shell
Without ids:  test_add_with_ids[1-2-3]         ← unreadable on failure
With ids:     test_add_with_ids[positive numbers] ← immediately clear
```

In [ ]:
# Dynamic IDs with pytest.param — most powerful approach
@pytest.mark.parametrize(
    "email, is_valid",
    [
        pytest.param("user@example.com",  True,  id="valid standard email"),
        pytest.param("user@.com",         False, id="missing domain label"),
        pytest.param("@example.com",      False, id="missing local part"),
        pytest.param("user@exam_ple.com", False, id="underscore in domain"),
    ]
)
def test_email_validation(email, is_valid):
    assert validate_email(email) == is_valid

**Level 3 — Stacking Decorators (Cartesian Product)**

In [ ]:
@pytest.mark.parametrize("role", ["admin", "moderator", "viewer"])
@pytest.mark.parametrize("method", ["GET", "POST", "DELETE"])
def test_endpoint_access(method, role):
    # pytest generates 3 × 3 = 9 test combinations automatically
    ...

# Generated tests:
# test_endpoint_access[GET-admin]
# test_endpoint_access[GET-moderator]
# test_endpoint_access[GET-viewer]
# test_endpoint_access[POST-admin]
# ... and so on

**Level 4 — `pytest.param` with Marks (skip, xfail)**

In [ ]:
@pytest.mark.parametrize(
    "input_val, expected",
    [
        pytest.param(2,    4,   id="positive even"),
        pytest.param(3,    9,   id="positive odd"),
        pytest.param(-2,   4,   id="negative even"),
        pytest.param(
            None, None,
            marks=pytest.mark.skip(reason="None handling not implemented yet"),
            id="none input"
        ),
        pytest.param(
            float("inf"), float("inf"),
            marks=pytest.mark.xfail(reason="known overflow bug #123", strict=True),
            id="infinity"
        ),
    ]
)
def test_square(input_val, expected):
    assert input_val ** 2 == expected

```shell
Output:
  PASSED  test_square[positive even]
  PASSED  test_square[positive odd]
  PASSED  test_square[negative even]
  SKIPPED test_square[none input]      ← skipped with reason
  XFAIL   test_square[infinity]        ← expected to fail
```

---

**Level 5 — Indirect Parametrize (using fixtures)**

In [ ]:
# conftest.py
import pytest

@pytest.fixture
def user_with_role(request):
    """
    Fixture that creates a user with the role passed via indirect.
    request.param holds the value from parametrize.
    """
    role = request.param
    return {"id": 1, "username": "testuser", "role": role}


# test_access.py
@pytest.mark.parametrize(
    "user_with_role, can_delete",
    [
        ("admin",     True),
        ("moderator", False),
        ("viewer",    False),
    ],
    indirect=["user_with_role"],  # ← tells pytest: pass through fixture, not raw value
)
def test_delete_permission(user_with_role, can_delete):
    # user_with_role is now the dict from the fixture, not the string "admin"
    result = check_delete_permission(user_with_role)
    assert result == can_delete

**2. `pytest.raises` — Context Manager Deep Dive**

In [ ]:
import pytest

# ── Basic usage ───────────────────────────────────────────────
def test_division_by_zero():
    with pytest.raises(ZeroDivisionError):
        1 / 0
    # code AFTER the with block runs normally
    # if NO exception → test FAILS
    # if WRONG exception → test FAILS and shows what was raised instead


# ── Inspecting the exception object ──────────────────────────
def test_exception_message():
    with pytest.raises(ValueError) as exc_info:
        # exc_info is ExceptionInfo — a pytest wrapper around the exception
        raise ValueError("price must be positive, got -5")

    # exc_info is populated AFTER the with block exits
    assert "price must be positive" in str(exc_info.value)
    assert exc_info.type is ValueError
    assert exc_info.value.args[0] == "price must be positive, got -5"


# ── match= parameter — regex on the message ──────────────────
def test_exception_match():
    with pytest.raises(ValueError, match=r"price must be positive, got -\d+"):
        raise ValueError("price must be positive, got -5")
    # match uses re.search() under the hood — partial match is enough

**3. Power Combo — `parametrize` + `pytest.raises`**

This is the pattern that impresses interviewers — testing both valid and invalid inputs in one parametrized test:

In [ ]:
# service.py — what we're testing
class OrderService:
    def create_order(self, user_id: int, amount: float, description: str):
        if not isinstance(user_id, int) or user_id <= 0:
            raise ValueError(f"Invalid user_id: {user_id}")
        if amount <= 0:
            raise ValueError(f"Amount must be positive, got {amount}")
        if not description.strip():
            raise ValueError("Description cannot be empty")
        return {"user_id": user_id, "amount": amount, "description": description}

In [ ]:
# test_order_service.py
import pytest
from service import OrderService

service = OrderService()

# ── Pattern 1: separate valid / invalid parametrize tests ─────

@pytest.mark.parametrize("user_id, amount, description, expected", [
    pytest.param(1,  99.99, "MacBook Pro",  {"user_id": 1, "amount": 99.99, "description": "MacBook Pro"}, id="valid standard"),
    pytest.param(99, 0.01,  "Cheap item",   {"user_id": 99, "amount": 0.01, "description": "Cheap item"},  id="valid minimal amount"),
])
def test_create_order_valid(user_id, amount, description, expected):
    result = service.create_order(user_id, amount, description)
    assert result == expected


@pytest.mark.parametrize("user_id, amount, description, error_type, error_match", [
    pytest.param(0,   99.99, "Item",  ValueError, r"Invalid user_id",           id="zero user_id"),
    pytest.param(-1,  99.99, "Item",  ValueError, r"Invalid user_id",           id="negative user_id"),
    pytest.param(1,   0,     "Item",  ValueError, r"Amount must be positive",   id="zero amount"),
    pytest.param(1,   -5.0,  "Item",  ValueError, r"Amount must be positive",   id="negative amount"),
    pytest.param(1,   10.0,  "   ",   ValueError, r"Description cannot be empty", id="blank description"),
    pytest.param(1,   10.0,  "",      ValueError, r"Description cannot be empty", id="empty description"),
])
def test_create_order_invalid(user_id, amount, description, error_type, error_match):
    with pytest.raises(error_type, match=error_match):
        service.create_order(user_id, amount, description)

**4. Real-World Django + FastAPI Test Examples**

Django — Testing View Access Control

In [ ]:
# tests/test_views.py
import pytest
from django.test import Client
from django.contrib.auth.models import User, Group


@pytest.fixture
def client():
    return Client()

@pytest.fixture
def make_user(db):
    """Factory fixture — creates users with any role."""
    def _make(username, role):
        user = User.objects.create_user(username=username, password="pass")
        group, _ = Group.objects.get_or_create(name=role)
        user.groups.add(group)
        return user
    return _make


@pytest.mark.parametrize("role, url, method, expected_status", [
    # admin can do everything
    pytest.param("admin", "/api/users/1/delete/", "delete", 200, id="admin-delete"),
    pytest.param("admin", "/api/posts/",          "get",    200, id="admin-list"),

    # moderator can moderate, cannot delete users
    pytest.param("moderator", "/api/posts/1/moderate/", "post", 200, id="moderator-moderate"),
    pytest.param("moderator", "/api/users/1/delete/",   "delete", 403, id="moderator-no-delete"),

    # viewer is read-only
    pytest.param("viewer", "/api/posts/",           "get",  200, id="viewer-list"),
    pytest.param("viewer", "/api/posts/1/moderate/","post", 403, id="viewer-no-moderate"),
    pytest.param("viewer", "/api/users/1/delete/",  "delete", 403, id="viewer-no-delete"),
])
def test_role_based_access(client, make_user, role, url, method, expected_status):
    user = make_user(f"{role}_user", role)
    client.force_login(user)
    response = getattr(client, method)(url)  # client.get / client.post / client.delete
    assert response.status_code == expected_status

FastAPI — Testing Your require_role Dependency

In [ ]:
# tests/test_dependencies.py
import pytest
from fastapi.testclient import TestClient
from unittest.mock import patch
from main import app
from models import User, UserGroupEnum


client = TestClient(app)

def make_user(groups: list[UserGroupEnum]) -> User:
    return User(id=1, username="testuser", groups=groups)


@pytest.mark.parametrize("user_groups, endpoint, expected_status, expected_detail", [
    pytest.param(
        [UserGroupEnum.ADMIN], "/posts/1", 200, None,
        id="admin-can-delete"
    ),
    pytest.param(
        [UserGroupEnum.MODERATOR], "/posts/1", 403, "Required role(s): admin",
        id="moderator-cannot-delete"
    ),
    pytest.param(
        [UserGroupEnum.VIEWER], "/posts/1", 403, "Required role(s): admin",
        id="viewer-cannot-delete"
    ),
    pytest.param(
        [UserGroupEnum.ADMIN, UserGroupEnum.MODERATOR], "/posts/1", 200, None,
        id="admin-moderator-can-delete"
    ),
])
def test_require_role_dependency(
    user_groups, endpoint, expected_status, expected_detail
):
    mock_user = make_user(user_groups)

    # patch get_current_user so we don't need a real JWT
    with patch("dependencies.get_current_user", return_value=mock_user):
        response = client.delete(endpoint)

    assert response.status_code == expected_status
    if expected_detail:
        assert expected_detail in response.json()["detail"]

**5. `pytest.warns` and `pytest.approx` — Complete the Picture**

In [ ]:
import warnings
import pytest


# ── pytest.warns ──────────────────────────────────────────────
def legacy_divide(a, b):
    if b == 0:
        warnings.warn("Division by zero, returning 0", DeprecationWarning)
        return 0
    return a / b

@pytest.mark.parametrize("a, b, expected_warning", [
    pytest.param(10, 0, DeprecationWarning, id="zero divisor triggers warning"),
])
def test_legacy_divide_warns(a, b, expected_warning):
    with pytest.warns(expected_warning, match="Division by zero"):
        result = legacy_divide(a, b)
    assert result == 0


# ── pytest.approx — float comparisons ────────────────────────
@pytest.mark.parametrize("a, b, expected", [
    pytest.param(0.1, 0.2, 0.3, id="classic float trap"),
    pytest.param(1/3, 1/3, 2/3, id="repeating decimal"),
])
def test_float_addition(a, b, expected):
    # assert 0.1 + 0.2 == 0.3  → FAILS (0.30000000000000004)
    assert a + b == pytest.approx(expected, rel=1e-9)  # passes ✅

**Mental Model — One Summary**

```shell
@pytest.mark.parametrize("x, y", [...])   DECORATOR
   │
   └── drives the test function with each row as arguments
         │
         ├── pytest.raises(ErrorType) as exc_info    CONTEXT MANAGER
         │      └── asserts exception IS raised, lets you inspect it
         │
         ├── pytest.warns(WarningType)               CONTEXT MANAGER
         │      └── asserts warning IS issued
         │
         └── pytest.approx(value)                    HELPER
                └── float-safe equality comparison


pytest.param(...)  →  wraps individual rows to add: id=, marks=
indirect=[...]     →  routes param through a fixture instead of raw value
```

The interview move: "I parametrize happy paths and error paths together using pytest.param with marks= for skips and xfail. For exception testing I pass the exception type and match pattern as parameters and use pytest.raises as a context manager inside the test body — this keeps all cases in one place and makes the test matrix obvious at a glance."